In [ ]:
# ==============================================================================
# 1. LIBRARIES
# ==============================================================================
library(Seurat)
library(dplyr)
library(jsonlite)
library(tiff)


# ==============================================================================
# 3. PATHS & EXECUTION
# ==============================================================================
base_path <- "/mnt/home3/miska/nm667/scratch/inProgress/dev/data/Kidney_ST/ST_image_tif_data/"

# File definitions
rds_path       <- paste0(base_path, "GSE211785_EXPORT_ST_counts.rds")
meta_path      <- paste0(base_path, "GSE211785_ST_metadata.txt")
tif_path       <- paste0(base_path, "GSE211785_V11Y24-076_B1-HK2852.tif")
json_path      <- paste0(base_path, "GSE211785_V11Y24-076-B1-HK2852.json")
whitelist_path <- paste0(base_path, "visium-v1_coordinates.txt")

# --- Step 1: Create Object ---
raw_counts <- readRDS(rds_path)
seurat_obj <- CreateSeuratObject(counts = raw_counts)

# --- Step 2: Add Metadata ---
metadata_supp <- read.delim(meta_path, sep = "\t", header = TRUE, row.names = 1)
seurat_obj <- AddMetaData(seurat_obj, metadata = metadata_supp)





In [ ]:
library(jsonlite)
library(dplyr)

# ==============================================================================
# 1. DEFINE PATHS (Using your provided variables)
# ==============================================================================
base_path      <- "/mnt/home3/miska/nm667/scratch/inProgress/dev/data/Kidney_ST/ST_image_tif_data/"
json_file      <- paste0(base_path, "GSE211785_V11Y24-076-B1-HK2852.json")
whitelist_path <- paste0(base_path, "visium-v1_coordinates.txt")

# ==============================================================================
# 2. LOAD AND MERGE DATA
# ==============================================================================

# A. Load the JSON Mapping (Pixel positions)
# NOTE: use $oligo, NOT $fiducial.
#   $fiducial = 555 alignment-frame markers (hourglass etc.), own grid: row 0-58, col 0-99
#   $oligo    = 4992 capture spots,                           grid: row 0-77, col 0-127
# Joining on $fiducial only matches 479/4992 barcodes -> ~90% NA pixel coordinates.
json_data <- fromJSON(json_file)
mapping <- as.data.frame(json_data$oligo) %>%
  select(row, col, imageX, imageY) %>%
  distinct()

# B. Load the Whitelist (Grid positions)
# Note: Based on your file snippet, Col 1=Barcode, Col 2=X (col), Col 3=Y (row)
coords <- read.table(whitelist_path, header = FALSE, stringsAsFactors = FALSE)
colnames(coords) <- c("barcode", "array_col", "array_row")

# The whitelist is 1-based (col 1-128, row 1-78); the JSON oligo grid is 0-based
# (col 0-127, row 0-77). Shift to 0-based so the join lands on all 4992 spots.
coords <- coords %>%
  mutate(array_col = array_col - 1L,
         array_row = array_row - 1L)

# C. Create the Standard Table
# We map array_row/col to the JSON's row/col to get the image pixels
tissue_positions <- coords %>%
  left_join(mapping, by = c("array_row" = "row", "array_col" = "col")) %>%
  mutate(in_tissue = 1) %>% # We assume all barcodes in your whitelist are in tissue
  select(
    barcode, 
    in_tissue, 
    array_row, 
    array_col, 
    pxl_row_in_fullres = imageY, 
    pxl_col_in_fullres = imageX
  )

# Fail loudly rather than silently writing NA coordinates
n_missing <- sum(is.na(tissue_positions$pxl_col_in_fullres))
cat("Spots mapped to pixels:", nrow(tissue_positions) - n_missing, "/", nrow(tissue_positions), "\n")
stopifnot(n_missing == 0)

# D. Save the file in 10x format (No header, comma-separated)
output_path <- paste0(base_path, "tissue_positions.csv")
write.table(tissue_positions, 
            file = output_path, 
            sep = ",", 
            col.names = FALSE, 
            row.names = FALSE, 
            quote = FALSE)

cat("✅ Success! Created tissue_positions.csv at:", output_path, "\n")

In [ ]:
# 1. Get Barcodes from all three sources
rds_barcodes  <- colnames(raw_counts) # From the RDS matrix
meta_barcodes <- rownames(metadata_supp) # From the metadata file
coord_data    <- read.table(whitelist_path, header = FALSE, stringsAsFactors = FALSE)
coord_barcodes <- coord_data$V1 # First column of your whitelist

# 2. Create "Clean" versions (removing the -1_1 suffix)
rds_clean   <- substr(rds_barcodes, 1, 16)
meta_clean  <- substr(meta_barcodes, 1, 16)
# coord_barcodes are already clean (16 chars) based on your description

# 3. generate the Report
message("--- BARCODE COUNT REPORT ---")
print(paste0("Total in RDS: ", length(rds_barcodes)))
print(paste0("Total in Metadata: ", length(meta_barcodes)))
print(paste0("Total in Whitelist: ", length(coord_barcodes)))

message("\n--- MATCHING ANALYSIS (RAW) ---")
print(paste0("RDS exactly matches Metadata: ", sum(rds_barcodes %in% meta_barcodes)))
print(paste0("RDS exactly matches Whitelist: ", sum(rds_barcodes %in% coord_barcodes)))

message("\n--- MATCHING ANALYSIS (CLEANED) ---")
print(paste0("Cleaned RDS matches Whitelist: ", sum(rds_clean %in% coord_barcodes)))

# 4. Identify the "Mismatched" spots (Optional: look at the first few)
mismatched <- rds_clean[!(rds_clean %in% coord_barcodes)]
if(length(mismatched) > 0) {
  message("\n--- SAMPLE OF MISMATCHED BARCODES ---")
  print(head(mismatched))
} else {
  message("\n✅ SUCCESS: All RDS barcodes found in Whitelist!")
}

In [ ]:
# 1. Identify the subset in your Metadata
# Assuming 'orig.ident' or a similar column identifies the sample
sample_id <- "HK2852_ST" 
subset_meta <- metadata_supp[metadata_supp$orig.ident == sample_id, ]

# 2. Get the barcodes for this specific subset
rds_subset_barcodes <- rownames(subset_meta) # These are the "AAAC...-1_1" names
coord_data <- read.table(whitelist_path, header = FALSE, stringsAsFactors = FALSE)
coord_barcodes <- coord_data$V1 # These are the "AAAC..." names

# 3. Clean the subset barcodes for matching
rds_subset_clean <- substr(rds_subset_barcodes, 1, 16)

# 4. Generate Subset Report
message(paste0("--- DIAGNOSTIC FOR SAMPLE: ", sample_id, " ---"))
cat("Barcodes in Metadata for this sample: ", nrow(subset_meta), "\n")
cat("Barcodes in Whitelist (Total Slide):  ", length(coord_barcodes), "\n")

# 5. Check Intersection
matches <- sum(rds_subset_clean %in% coord_barcodes)
pct_match <- (matches / length(rds_subset_clean)) * 100

cat("Successfully matched to Coordinates: ", matches, "\n")
cat("Percentage Match:                   ", round(pct_match, 2), "%\n")

if(matches == 0) {
  warning("❌ ZERO matches found. Check if the whitelist version matches the slide.")
} else if (matches < nrow(subset_meta)) {
  message("⚠️ Some barcodes in the RDS are missing from the coordinate file.")
} else {
  message("✅ Perfect match for the subset!")
}

In [ ]:
# ==============================================================================
# 1. LIBRARIES
# ==============================================================================
library(Seurat)
library(dplyr)
library(jsonlite)
library(tiff)

# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================

#' get_spot_coord: Maps whitelist grid coordinates to image pixels using JSON
get_spot_coord <- function(json_path, whitelist_path) {
  # Load the JSON mapping table
  json_data <- fromJSON(json_path)
  
  # Extract the direct pixel mapping from the 'oligo' array
  # This maps [row, col] -> [imageX, imageY] for the 4992 capture spots.
  # (NOT 'fiducial' - that is the 555-marker alignment frame on a different grid.)
  mapping_table <- as.data.frame(json_data$oligo) %>%
    select(row, col, imageX, imageY) %>%
    distinct()
  
  # Load the Whitelist (3 columns: Barcode, Grid_X, Grid_Y)
  coords <- read.table(whitelist_path, header = FALSE, stringsAsFactors = FALSE)
  colnames(coords) <- c("barcode", "col", "row")
  
  # Whitelist is 1-based, JSON oligo grid is 0-based -> shift before joining
  coords <- coords %>% mutate(col = col - 1L, row = row - 1L)
  
  # Join the barcode list with the pixel positions
  final_coords <- coords %>%
    left_join(mapping_table, by = c("row", "col"))
  
  stopifnot(!any(is.na(final_coords$imageX)))
  
  rownames(final_coords) <- final_coords$barcode
  return(final_coords)
}

#' get_image: Loads TIF as a raw 3D array (Required for newer Seurat)
get_image <- function(img_path) {
  img <- readTIFF(img_path)
  # Newer Seurat versions require a raw array [H, W, C], not a 'raster' object
  return(list(img = img, scalef = 1.0))
}

#' add_image_seurat_standard: The main "Stitcher" function
add_image_seurat_standard <- function(so, img_path, json_path, whitelist_path, slice_name = "slice1") {
  
  # 1. Get Coordinate Mapping
  coord_data <- get_spot_coord(json_path, whitelist_path)
  
  # 2. Solve Barcode Mismatch
  # Seurat: "AAAC...-1_1" | Whitelist: "AAAC..."
  cells_seurat <- colnames(so)
  cells_clean  <- substr(cells_seurat, 1, 16) 
  names(cells_clean) <- cells_seurat          
  
  # Match coordinates using clean names, then restore full names
  matched_coords <- coord_data[cells_clean, ]
  rownames(matched_coords) <- names(cells_clean)
  
  # 3. Format for Seurat VisiumV1 Class
  tissue_positions <- data.frame(
    tissue = 1,
    row = matched_coords$row,
    col = matched_coords$col,
    imagerow = matched_coords$imageY,
    imagecol = matched_coords$imageX
  )
  rownames(tissue_positions) <- rownames(matched_coords)
  
  # 4. Load Image Data (full resolution -> scale factors are 1)
  img_info <- get_image(img_path)
  
  # 5. Create the formal VisiumV1 Object
  # scale.factors$spot / $fiducial are DIAMETERS in full-res pixels (from the JSON),
  # not scaling ratios. $hires / $lowres are the ratios; here the stored image is
  # full-res, so both are 1.
  json_data        <- fromJSON(json_path)
  spot_dia_fullres <- median(as.data.frame(json_data$oligo)$dia)
  fid_dia_fullres  <- median(as.data.frame(json_data$fiducial)$dia)
  
  visium_obj <- new(
    Class = "VisiumV1",
    # assay/key are required: new() leaves them empty, so DefaultAssay()
    # returns character(0) and `so[["slice"]] <- obj` errors out.
    assay = "RNA",
    key = paste0(slice_name, "_"),
    image = img_info$img,
    scale.factors = scalefactors(spot = spot_dia_fullres, fiducial = fid_dia_fullres,
                                 hires = 1, lowres = 1),
    coordinates = tissue_positions,
    spot.radius = fid_dia_fullres / max(dim(img_info$img))
  )
  
  # 6. Attach to Seurat
  so[[slice_name]] <- visium_obj
  return(so)
}

# ==============================================================================
# 3. MAIN WORKFLOW
# ==============================================================================

# --- Define Paths ---
base_path <- "/mnt/home3/miska/nm667/scratch/inProgress/dev/data/Kidney_ST/ST_image_tif_data/"

rds_file_path      <- paste0(base_path, "GSE211785_EXPORT_ST_counts.rds")
metadata_file_path <- paste0(base_path, "GSE211785_ST_metadata.txt")
target_sample      <- "HK2852_ST" 
tif_file           <- paste0(base_path, "GSE211785_V11Y24-076_B1-HK2852.tif")
json_file          <- paste0(base_path, "GSE211785_V11Y24-076-B1-HK2852.json")
whitelist_path     <- paste0(base_path, "visium-v1_coordinates.txt")

# --- A. Load Molecular Data ---
print("⏳ Loading Counts and Metadata...")
raw_counts <- readRDS(rds_file_path)
seurat_combined <- CreateSeuratObject(counts = raw_counts, project = "Kidney_ST")

metadata_supp <- read.delim(metadata_file_path, sep = "\t", header = TRUE, row.names = 1)
seurat_combined <- AddMetaData(seurat_combined, metadata = metadata_supp)

# --- B. Subset for the Target Image ---
# Very important: You only attach one image to the spots belonging to that image
print(paste0("⏳ Subsetting for ", target_sample, "..."))
so_subset <- subset(seurat_combined, subset = orig.ident == target_sample)

# --- C. Attach Spatial Data ---
# NOTE: this reads the TIF at full resolution (11350 x 10529 x 3 doubles ~ 2.9 GB)
# and is slow. See the final cell for the resized version.
print("⏳ Attaching Spatial Image and Coordinates...")
so_subset <- add_image_seurat_standard(
  so = so_subset,
  img_path = tif_file,
  json_path = json_file,
  whitelist_path = whitelist_path,
  slice_name = "HK2852"
)

# --- D. Final Plot ---
print("✅ Done! Generating Plot...")
SpatialDimPlot(so_subset, group.by = "celltype", pt.size.factor = 1.2) + 
  ggtitle(paste("Spatial Mapping:", target_sample))

In [ ]:
# ==============================================================================
# 1. LIBRARIES
# ==============================================================================
library(Seurat)
library(dplyr)
library(jsonlite)
library(magick) # Essential for high-speed image resizing

# ==============================================================================
# 2. FILE PATHS
# ==============================================================================
base_path <- "/mnt/home3/miska/nm667/scratch/inProgress/dev/data/Kidney_ST/ST_image_tif_data/"

rds_file_path      <- paste0(base_path, "GSE211785_EXPORT_ST_counts.rds")
metadata_file_path <- paste0(base_path, "GSE211785_ST_metadata.txt")
target_sample      <- "HK2852_ST" 
tif_file           <- paste0(base_path, "GSE211785_V11Y24-076_B1-HK2852.tif")
json_file          <- paste0(base_path, "GSE211785_V11Y24-076-B1-HK2852.json")
whitelist_path     <- paste0(base_path, "visium-v1_coordinates.txt")

# ==============================================================================
# 3. STEP 1: RESIZE IMAGE & GET SCALE FACTOR
# ==============================================================================
print("⏳ Resizing image for performance...")
img_magick     <- image_read(tif_file)
original_info  <- image_info(img_magick)
original_width <- original_info$width

# Resize to 2000px width (Standard Seurat Hires size)
img_small    <- image_resize(img_magick, "2000x")
scaled_width <- image_info(img_small)$width

# Calculate the critical scale factor
scale_factor <- scaled_width / original_width

# Convert magick image to numeric array for Seurat.
# as.numeric() on its own DROPS the dim attribute and returns a flat vector,
# which Seurat cannot plot. Keep the [height, width, channels] shape.
img_raw   <- img_small[[1]]
img_array <- array(as.numeric(img_raw) / 255,
                   dim = c(dim(img_raw)[3], dim(img_raw)[2], 3))

# ==============================================================================
# 4. STEP 2: GENERATE COORDINATES (tissue_positions.csv style)
# ==============================================================================
print("⏳ Mapping coordinates...")
json_data <- fromJSON(json_file)

# $oligo = the 4992 capture spots. $fiducial = the alignment frame (wrong table).
mapping   <- as.data.frame(json_data$oligo) %>%
  select(row, col, imageX, imageY) %>%
  distinct()

coords <- read.table(whitelist_path, header = FALSE, stringsAsFactors = FALSE)
colnames(coords) <- c("barcode", "array_col", "array_row")

# Whitelist is 1-based, JSON oligo grid is 0-based
coords <- coords %>%
  mutate(array_col = array_col - 1L,
         array_row = array_row - 1L)

# Merge grid positions with pixel positions from JSON
tissue_positions <- coords %>%
  left_join(mapping, by = c("array_row" = "row", "array_col" = "col")) %>%
  mutate(in_tissue = 1) %>%
  select(
    barcode, 
    in_tissue, 
    array_row, 
    array_col, 
    pxl_row_in_fullres = imageY, 
    pxl_col_in_fullres = imageX
  )

stopifnot(!any(is.na(tissue_positions$pxl_col_in_fullres)))

# Spot / fiducial DIAMETERS in full-res pixels, straight from the JSON
spot_dia_fullres <- median(as.data.frame(json_data$oligo)$dia)
fid_dia_fullres  <- median(as.data.frame(json_data$fiducial)$dia)

# ==============================================================================
# 5. STEP 3: ASSEMBLE SEURAT OBJECT
# ==============================================================================
print("⏳ Building Seurat object...")
# Load counts and subset
raw_counts <- readRDS(rds_file_path)
so_full    <- CreateSeuratObject(counts = raw_counts)
metadata   <- read.delim(metadata_file_path, row.names = 1)
so_full    <- AddMetaData(so_full, metadata = metadata)
so_subset  <- subset(so_full, subset = orig.ident == target_sample)

# Barcode Matching (Fixing the -1_1 suffix)
cells_seurat <- colnames(so_subset)
cells_clean  <- substr(cells_seurat, 1, 16)
rownames(tissue_positions) <- tissue_positions$barcode
matched_coords <- tissue_positions[cells_clean, ]
rownames(matched_coords) <- cells_seurat

# Create VisiumV1 Object.
# spot / fiducial   = diameters in full-res pixels (NOT scaling ratios)
# hires / lowres    = the ratio between the stored image and full-res.
#   The image stored here IS the 2000px resize, so both equal scale_factor.
#   Coordinates stay in FULL-RES pixels; Seurat multiplies them by $lowres at plot time.
visium_obj <- new(
  Class = "VisiumV1",
  # assay/key are required: new() leaves them empty, so DefaultAssay()
  # returns character(0) and `so[["slice"]] <- obj` errors out.
  assay = "RNA",
  key = "HK2852_",
  image = img_array,
  scale.factors = scalefactors(
    spot     = spot_dia_fullres, 
    fiducial = fid_dia_fullres, 
    hires    = scale_factor, 
    lowres   = scale_factor
  ),
  coordinates = data.frame(
    tissue = matched_coords$in_tissue,
    row = matched_coords$array_row,
    col = matched_coords$array_col,
    imagerow = matched_coords$pxl_row_in_fullres,
    imagecol = matched_coords$pxl_col_in_fullres,
    row.names = cells_seurat
  ),
  spot.radius = fid_dia_fullres * scale_factor / max(dim(img_array))
)

so_subset[["HK2852"]] <- visium_obj

# ==============================================================================
# 6. STEP 4: CLEANUP & PLOT
# ==============================================================================
rm(so_full, raw_counts, img_magick, img_small, img_raw)
gc()

print("✅ Success! Generating Plot...")
SpatialDimPlot(so_subset, group.by = "celltype", pt.size.factor = 1.3, alpha = 1)

In [ ]:
# ==============================================================================
# OPTIMIZED STEP 3: PRE-SUBSETTING & FAST ASSEMBLY
# (reuses tissue_positions / img_array / scale_factor / *_dia_fullres from the cell above)
# ==============================================================================
print("⏳ Loading and Pre-subsetting Counts...")
raw_counts <- readRDS(rds_file_path)
metadata   <- read.delim(metadata_file_path, row.names = 1)

# Find the barcodes that belong to HK2852_ST in the metadata
hk2852_barcodes <- rownames(metadata)[metadata$orig.ident == target_sample]

# Subset the matrix and metadata BEFORE making the Seurat Object
# This reduces the data size from 37,000 spots to ~2,200 immediately
counts_sub   <- raw_counts[, hk2852_barcodes]
metadata_sub <- metadata[hk2852_barcodes, ]

print("⏳ Building Small Seurat object...")
so_subset <- CreateSeuratObject(counts = counts_sub, meta.data = metadata_sub)

# ==============================================================================
# OPTIMIZED STEP 5: ATTACHING SPATIAL DATA
# ==============================================================================
# Map the barcodes (Handling the -1_1 suffix)
cells_seurat <- colnames(so_subset)
cells_clean  <- substr(cells_seurat, 1, 16)

# Use fast vector indexing instead of data.frame joining
rownames(tissue_positions) <- tissue_positions$barcode
matched_coords <- tissue_positions[cells_clean, ]
stopifnot(!any(is.na(matched_coords$pxl_col_in_fullres)))

# Build the spatial data frame directly
spatial_df <- data.frame(
    tissue   = 1,
    row      = matched_coords$array_row,
    col      = matched_coords$array_col,
    imagerow = matched_coords$pxl_row_in_fullres,
    imagecol = matched_coords$pxl_col_in_fullres,
    row.names = cells_seurat
)

# Create the VisiumV1 object
# spot / fiducial = full-res DIAMETERS; hires / lowres = the resize ratio.
visium_obj <- new(
  Class = "VisiumV1",
  # assay/key are required: new() leaves them empty, so DefaultAssay()
  # returns character(0) and `so[["slice"]] <- obj` errors out.
  assay = "RNA",
  key = "HK2852_",
  image = img_array,
  scale.factors = scalefactors(
    spot     = spot_dia_fullres, 
    fiducial = fid_dia_fullres, 
    hires    = scale_factor, 
    lowres   = scale_factor
  ),
  coordinates = spatial_df,
  spot.radius = fid_dia_fullres * scale_factor / max(dim(img_array))
)

so_subset[["HK2852"]] <- visium_obj

In [ ]:
# ==============================================================================
# 1. LIBRARIES
# ==============================================================================
library(Seurat)
library(dplyr)
library(jsonlite)
library(magick) # For fast resizing
library(tiff)   # For fast array reading

# ==============================================================================
# 2. FILE PATHS
# ==============================================================================
base_path <- "/mnt/home3/miska/nm667/scratch/inProgress/dev/data/Kidney_ST/ST_image_tif_data/"

rds_file_path      <- paste0(base_path, "GSE211785_EXPORT_ST_counts.rds")
metadata_file_path <- paste0(base_path, "GSE211785_ST_metadata.txt")
target_sample      <- "HK2852_ST" 
tif_file           <- paste0(base_path, "GSE211785_V11Y24-076_B1-HK2852.tif")
json_file          <- paste0(base_path, "GSE211785_V11Y24-076-B1-HK2852.json")
whitelist_path     <- paste0(base_path, "visium-v1_coordinates.txt")
out_rds            <- paste0(base_path, "processed_spatial_objects/HK2852_ST_spatial_v2.rds")

# ==============================================================================
# 3. FAST IMAGE PROCESSING (The bottleneck fix)
# ==============================================================================
print("⏳ Resizing image via fast-path...")
img_magick     <- image_read(tif_file)
original_width <- image_info(img_magick)$width   # 11350 for this slide

# Resize to 2000px (Standard Hi-res)
img_small    <- image_resize(img_magick, "2000x")
scale_factor <- image_info(img_small)$width / original_width   # ~0.1762

# Fast Array Conversion: Save to temp and read back (much faster than as.numeric).
# readTIFF returns a [height, width, channels] array scaled to 0-1, which is what
# Seurat wants. Do NOT use as.numeric() here - it drops the dim attribute.
tmp <- tempfile(fileext = ".tif")
image_write(img_small, tmp, format = "tiff")
img_array <- readTIFF(tmp)
file.remove(tmp) # Cleanup
rm(img_magick, img_small)

# ==============================================================================
# 4. COORDINATE MAPPING
# ==============================================================================
print("⏳ Preparing coordinates...")
json_data <- fromJSON(json_file)

# Use $oligo - the 4992 capture spots (grid: row 0-77, col 0-127).
# $fiducial is the 555-marker alignment frame on its own grid (row 0-58, col 0-99);
# joining against it matches only 479/4992 barcodes and leaves ~90% NA coordinates.
mapping   <- as.data.frame(json_data$oligo) %>%
  select(row, col, imageX, imageY) %>%
  distinct()

coords <- read.table(whitelist_path, header = FALSE)
colnames(coords) <- c("barcode", "array_col", "array_row")

# The whitelist is 1-based (col 1-128, row 1-78), the JSON grid is 0-based.
# Without this shift the join lands on 4890/4992 instead of 4992/4992.
coords <- coords %>%
  mutate(array_col = array_col - 1L,
         array_row = array_row - 1L)

tissue_positions <- coords %>%
  left_join(mapping, by = c("array_row" = "row", "array_col" = "col")) %>%
  mutate(in_tissue = 1)

cat("Spots mapped:", sum(!is.na(tissue_positions$imageX)), "/", nrow(tissue_positions), "\n")
stopifnot(!any(is.na(tissue_positions$imageX)))

# Spot / fiducial DIAMETERS in full-res pixels, read straight from the JSON
spot_dia_fullres <- median(as.data.frame(json_data$oligo)$dia)      # ~86.1
fid_dia_fullres  <- median(as.data.frame(json_data$fiducial)$dia)   # ~139.2

# ==============================================================================
# 5. ASSEMBLE SEURAT OBJECT (Optimized for Speed)
# ==============================================================================
print("⏳ Subsetting and assembling object...")
raw_counts <- readRDS(rds_file_path)
metadata   <- read.delim(metadata_file_path, row.names = 1)

# Pre-subset matrix to keep object light
hk2852_cells <- rownames(metadata)[metadata$orig.ident == target_sample]
so_subset    <- CreateSeuratObject(counts = raw_counts[, hk2852_cells], 
                                   meta.data = metadata[hk2852_cells, ])
rm(raw_counts); gc()

# Align barcodes
cells_seurat <- colnames(so_subset)
cells_clean  <- substr(cells_seurat, 1, 16)
rownames(tissue_positions) <- tissue_positions$barcode
matched_coords <- tissue_positions[cells_clean, ]
stopifnot(!any(is.na(matched_coords$imageX)))

# Build lean spatial dataframe (coordinates stay in FULL-RES pixel space -
# Seurat rescales them by scale.factors$lowres at plot time)
spatial_df <- data.frame(
  tissue   = 1,
  row      = as.integer(matched_coords$array_row),
  col      = as.integer(matched_coords$array_col),
  imagerow = as.numeric(matched_coords$imageY), # Pixel Y
  imagecol = as.numeric(matched_coords$imageX), # Pixel X
  row.names = cells_seurat
)

# Create VisiumV1 object.
#   spot / fiducial = DIAMETERS in full-res pixels (these are not scaling ratios)
#   hires / lowres  = ratio of the stored image to full-res. The image held here is
#                     the 2000px resize, so both are scale_factor. Using
#                     scale_factor * 0.3 for lowres shrinks the spots to 30% of
#                     their true position and bunches them in the top-left corner.
visium_obj <- new(
  Class = "VisiumV1",
  # assay/key are required: new() leaves them empty, so DefaultAssay()
  # returns character(0) and `so[["slice"]] <- obj` errors out.
  assay = "RNA",
  key = "HK2852_",
  image = img_array,
  scale.factors = scalefactors(spot = spot_dia_fullres, fiducial = fid_dia_fullres,
                               hires = scale_factor, lowres = scale_factor),
  coordinates = spatial_df,
  spot.radius = fid_dia_fullres * scale_factor / max(dim(img_array))
)

so_subset[["HK2852"]] <- visium_obj

# Save so the object does not have to be rebuilt every session
dir.create(dirname(out_rds), showWarnings = FALSE, recursive = TRUE)
saveRDS(so_subset, out_rds)
cat("💾 Saved:", out_rds, "\n")

# ==============================================================================
# 6. PLOT (Fastest settings)
# ==============================================================================
print("✅ Success! Plotting...")
# Use stroke = NA and pt.size.factor to speed up rendering
SpatialDimPlot(so_subset, group.by = "celltype", pt.size.factor = 1.3, stroke = NA)